In [71]:
import glob, os
import numpy as np
import pyxdf, mne
import torch
from sklearn.model_selection import train_test_split

In [72]:
# === ディレクトリパスを指定 ===
base_dir = r"E:\EEG_B4\sub-P001\ses-S002\eeg"

# === run-001 ～ run-003 の .xdf を自動抽出 ===
xdf_files = sorted(glob.glob(os.path.join(base_dir, "*run-0*_eeg.xdf")))
print(f"{len(xdf_files)} files found:")
for f in xdf_files:
    print(" -", os.path.basename(f))

7 files found:
 - sub-P001_ses-S002_task-MI_run-001_eeg.xdf
 - sub-P001_ses-S002_task-MI_run-002_eeg.xdf
 - sub-P001_ses-S002_task-MI_run-003_eeg.xdf
 - sub-P001_ses-S002_task-MI_run-004_eeg.xdf
 - sub-P001_ses-S002_task-MI_run-005_eeg.xdf
 - sub-P001_ses-S002_task-MI_run-006_eeg.xdf
 - sub-P001_ses-S002_task-MI_run-007_eeg.xdf


In [73]:
# === 共通設定 ===
CH_LABELS = ["FC1","FC2","C3","C4","CP1","CP2","P3","P4"]
SFREQ = 250
EVENT_ID = {"left": 1, "right": 2}
TMIN, TMAX = -0.5, 4.5
# === 前処理の設定 ===
APPLY_FILTERS   = True        # 学習時にフィルタしていたなら True
NOTCH_HZ        = 50.0        # 日本の商用電源
BANDPASS        = (1.0, 40.0) # MIなら 8–30Hz などに変えてもOK（学習時と合わせる）
REJECT_PTP_UV   = None         # 例: 150 ならピークtoピーク150µV超を自動除外。使わないなら None
DO_ZSCORE       = True        # 学習時の正規化に合わせる（epoch内z-scoreが無難）

In [74]:
all_x, all_y = [], []
for path in xdf_files:
    print(f"[INFO] Loading {os.path.basename(path)}")

    streams, header = pyxdf.load_xdf(
    path,
    synchronize_clocks=False,   # ★時計同期を無効化
    dejitter_timestamps=False,  # ★ジッタ補正も無効化
)


    eeg_stream = [s for s in streams if s["info"]["type"][0].lower() == "eeg"][0]
    data = eeg_stream["time_series"].T                 # (n_ch, n_samples) μV
    ts_eeg = eeg_stream["time_stamps"]

    marker_stream = [s for s in streams if s["info"]["type"][0].lower() == "markers"][0]
    markers = marker_stream["time_series"]
    ts_mrk = marker_stream["time_stamps"]

    # --- Raw生成（μV→V）
    info = mne.create_info(ch_names=CH_LABELS, sfreq=SFREQ, ch_types="eeg")
    raw = mne.io.RawArray(data * 1e-6, info, verbose=False)

    # --- フィルタ（学習時と同じに！）
    if APPLY_FILTERS:
        raw.notch_filter(NOTCH_HZ, fir_design="firwin", verbose=False)
        raw.filter(BANDPASS[0], BANDPASS[1], fir_design="firwin", verbose=False)

    # --- イベント配列
    events = []
    for lab, t in zip(markers, ts_mrk):
        label = lab[0] if isinstance(lab, (list, tuple, np.ndarray)) else lab
        if label not in EVENT_ID:
            continue
        fs = raw.info["sfreq"]
        # EEGタイムスタンプ基準でサンプルに変換
        sample = int((t - ts_eeg[0]) * fs)
        events.append([sample, 0, EVENT_ID[label]])
    events = np.array(events, dtype=int)
    if len(events) == 0:
        print(f"[WARN] no events in {os.path.basename(path)}")
        continue

    # --- （任意）アーティファクト除外しながらEpochs作成
    reject = None if REJECT_PTP_UV is None else dict(eeg=REJECT_PTP_UV * 1e-6)
    epochs = mne.Epochs(raw, events, event_id=EVENT_ID, tmin=TMIN, tmax=TMAX,
                        baseline=(-0.5,0.0), preload=True, reject=reject, verbose="ERROR")

    x = epochs.get_data()                # (n_trials, n_ch, n_times)  V単位
    y = epochs.events[:, 2] - 1

    all_x.append(x)
    all_y.append(y)


[INFO] Loading sub-P001_ses-S002_task-MI_run-001_eeg.xdf
[INFO] Loading sub-P001_ses-S002_task-MI_run-002_eeg.xdf


C:\Users\kgtk0\AppData\Local\Temp\ipykernel_13364\2722679931.py:49: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  x = epochs.get_data()                # (n_trials, n_ch, n_times)  V単位
C:\Users\kgtk0\AppData\Local\Temp\ipykernel_13364\2722679931.py:49: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  x = epochs.get_data()                # (n_trials, n_ch, n_times)  V単位


[INFO] Loading sub-P001_ses-S002_task-MI_run-003_eeg.xdf
[INFO] Loading sub-P001_ses-S002_task-MI_run-004_eeg.xdf


C:\Users\kgtk0\AppData\Local\Temp\ipykernel_13364\2722679931.py:49: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  x = epochs.get_data()                # (n_trials, n_ch, n_times)  V単位
C:\Users\kgtk0\AppData\Local\Temp\ipykernel_13364\2722679931.py:49: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  x = epochs.get_data()                # (n_trials, n_ch, n_times)  V単位


[INFO] Loading sub-P001_ses-S002_task-MI_run-005_eeg.xdf
[INFO] Loading sub-P001_ses-S002_task-MI_run-006_eeg.xdf


C:\Users\kgtk0\AppData\Local\Temp\ipykernel_13364\2722679931.py:49: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  x = epochs.get_data()                # (n_trials, n_ch, n_times)  V単位
C:\Users\kgtk0\AppData\Local\Temp\ipykernel_13364\2722679931.py:49: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  x = epochs.get_data()                # (n_trials, n_ch, n_times)  V単位


[INFO] Loading sub-P001_ses-S002_task-MI_run-007_eeg.xdf


C:\Users\kgtk0\AppData\Local\Temp\ipykernel_13364\2722679931.py:49: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  x = epochs.get_data()                # (n_trials, n_ch, n_times)  V単位


In [75]:
fs = 250
tmin, tmax = -0.5, 4.5
expected_T = int(round((tmax - tmin) * fs)) + 1 

all_x_fixed, all_y_fixed, all_groups = [], [], []
for file_idx, (x, y) in enumerate(zip(all_x, all_y)):
    T = x.shape[2]
    if T >= expected_T:
        
        x = x[..., :expected_T]
    else:
         #まれに短い場合だけスキップ or 末尾パディング（必要なら）
        #pad = expected_T - T
        #x = np.pad(x, ((0,0),(0,0),(0,pad)), mode='edge')
        pass
    all_x_fixed.append(x)
    all_y_fixed.append(y)
    all_groups.append(np.full(len(y), file_idx, dtype=int))
x_all = np.concatenate(all_x_fixed, axis=0)
y_all = np.concatenate(all_y_fixed, axis=0)
print("all:", x_all.shape, y_all.shape)

all: (154, 8, 1251) (154,)


In [76]:
X_train, X_test, y_train, y_test = train_test_split(
    x_all, y_all, test_size=0.22, random_state=42, stratify=y_all
)

In [77]:
#学習データのクラスバランス調整
idx0 = np.where(y_train == 0)[0]
idx1 = np.where(y_train == 1)[0]
min_len = min(len(idx0), len(idx1))
idx0_ds = np.random.choice(idx0, min_len, replace=False)
idx1_ds = np.random.choice(idx1, min_len, replace=False)
balanced_idx = np.concatenate([idx0_ds, idx1_ds])

X_train = X_train[balanced_idx]
y_train = y_train[balanced_idx]

In [78]:
# 正規化
X_train = (X_train - X_train.mean(axis=2, keepdims=True)) / X_train.std(axis=2, keepdims=True)
X_test  = (X_test  - X_test.mean(axis=2, keepdims=True))  / X_test.std(axis=2, keepdims=True)
print(X_train.shape,X_test.shape)

(120, 8, 1251) (34, 8, 1251)


In [79]:
from torch.utils.data import Dataset

class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)  # (N, C, T)
        self.y = torch.tensor(y, dtype=torch.long)     # (N,)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [80]:
from torch.utils.data import DataLoader

train_dataset = EEGDataset(X_train, y_train)
test_dataset = EEGDataset(X_test, y_test)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [81]:
import torch.nn as nn
import torch.nn.functional as F

C, T = X_train.shape[1], X_train.shape[2]

class EEGNet(nn.Module):
    def __init__(self, num_classes=2, channels=C, samples=T):
        super(EEGNet, self).__init__()

        # 第1層: Temporal Convolution
        self.firstconv = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=(1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(8)
        )

        # 第2層: Depthwise Convolution (Spatial filter)
        self.depthwiseConv = nn.Sequential(
            nn.Conv2d(8, 16, kernel_size=(channels, 1), groups=8, bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(p=0.25)
        )

        # 第3層: Separable Convolution
        self.separableConv = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=(1, 16), padding=(0, 8), bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(p=0.25)
        )

        # 全結合層
        self.classify = nn.Linear(self._get_flattened_size(channels, samples), num_classes)

    def _get_flattened_size(self, channels, samples):
        x = torch.zeros(1, 1, channels, samples)
        x = self.firstconv(x)
        x = self.depthwiseConv(x)
        x = self.separableConv(x)
        return x.numel()

    def forward(self, x):
        x = x.unsqueeze(1)  # (B, 1, C, T)
        x = self.firstconv(x)
        x = self.depthwiseConv(x)
        x = self.separableConv(x)
        x = x.view(x.size(0), -1)
        return self.classify(x)

In [82]:
device = torch.device("cuda")

C, T = X_train.shape[1], X_train.shape[2]
model = EEGNet(num_classes=2, channels=C, samples=T).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2, weight_decay=1e-4)

# 20, 50 エポック後に gamma=0.1倍にする → 1e-2 → 1e-3 → 1e-4
from torch.optim.lr_scheduler import MultiStepLR
scheduler = MultiStepLR(optimizer, milestones=[20, 50], gamma=0.1)

scaler = None  # そのままでOK



In [83]:
from torch.nn.utils import clip_grad_norm_
from sklearn.metrics import confusion_matrix, classification_report

def train_one_epoch(model, loader, criterion, optimizer, device, max_grad_norm=None):
    model.train()
    tot_loss, tot_correct, tot = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        if max_grad_norm is not None:
            clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()

        b = y.size(0)
        tot_loss    += loss.item() * b
        tot_correct += (logits.argmax(1) == y).sum().item()
        tot         += b
    return tot_loss / tot, tot_correct / tot


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    tot_loss, tot_correct, tot = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        b = y.size(0)
        tot_loss += loss.item() * b
        tot_correct += (logits.argmax(1) == y).sum().item()
        tot += b
    return tot_loss/tot, tot_correct/tot

@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    y_trues, y_preds = [], []
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        y_pred = logits.argmax(1).cpu().numpy()
        y_trues.append(y.numpy())
        y_preds.append(y_pred)
    return np.concatenate(y_trues), np.concatenate(y_preds)

In [84]:
print("X_train:", X_train.shape)  # (N, C, T)


X_train: (120, 8, 1251)


In [85]:
def get_lr(optim):
    return optim.param_groups[0]['lr']

epochs = 100
best_test = 0.0
best_epoch = -1
best_cm = None

for ep in range(1, epochs+1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)  # いつもの
    te_loss, te_acc = evaluate(model, test_loader, criterion, device)

    print(f"[{ep:03d}/{epochs}] "
          f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
          f"test_loss={te_loss:.4f} test_acc={te_acc:.4f} | lr={get_lr(optimizer):.2e}")

    # ← エポックの最後にステップ。ep=20,50 が終わった“次の”エポックからLRが下がる
    scheduler.step()
    
    if ep == 50:
        torch.save(model.state_dict(), "lastepoch.pt")


    if te_acc > best_test:
        best_test = te_acc
        best_epoch = ep
        torch.save(model.state_dict(), "best.pt")
        y_true, y_pred = predict_all(model, test_loader, device)
        best_cm = confusion_matrix(y_true, y_pred, labels=[0,1])
print(f"\nBEST test_acc={best_test:.4f}")

[001/100] train_loss=0.7071 train_acc=0.5250 | test_loss=0.6929 test_acc=0.5000 | lr=1.00e-02
[002/100] train_loss=0.6412 train_acc=0.6167 | test_loss=0.6304 test_acc=0.7059 | lr=1.00e-02
[003/100] train_loss=0.5472 train_acc=0.7333 | test_loss=0.6376 test_acc=0.6765 | lr=1.00e-02
[004/100] train_loss=0.4361 train_acc=0.8083 | test_loss=0.6446 test_acc=0.7059 | lr=1.00e-02
[005/100] train_loss=0.3786 train_acc=0.8583 | test_loss=0.6321 test_acc=0.6765 | lr=1.00e-02
[006/100] train_loss=0.3666 train_acc=0.8667 | test_loss=0.6249 test_acc=0.6176 | lr=1.00e-02
[007/100] train_loss=0.3168 train_acc=0.9083 | test_loss=0.6007 test_acc=0.6176 | lr=1.00e-02
[008/100] train_loss=0.2856 train_acc=0.9083 | test_loss=0.5670 test_acc=0.6765 | lr=1.00e-02
[009/100] train_loss=0.2434 train_acc=0.9333 | test_loss=0.5680 test_acc=0.6471 | lr=1.00e-02
[010/100] train_loss=0.2118 train_acc=0.9500 | test_loss=0.5801 test_acc=0.6471 | lr=1.00e-02
[011/100] train_loss=0.2230 train_acc=0.9083 | test_loss=0.5

In [86]:
# 念のため、保存したベストモデルを読み直して評価
best_state = torch.load("best.pt", map_location=device)
model.load_state_dict(best_state)

y_true, y_pred = predict_all(model, test_loader, device)
cm = confusion_matrix(y_true, y_pred, labels=[0,1])
cm_norm = confusion_matrix(y_true, y_pred, labels=[0,1], normalize="true")

print(f"\nBEST test_acc={best_test:.4f} (epoch {best_epoch})")
print("Confusion matrix (counts):\n", cm)
print("Confusion matrix (row-normalized):\n", np.round(cm_norm, 3))

# 二値なら派生指標も
tn, fp, fn, tp = cm.ravel()
precision = tp / (tp + fp + 1e-9)
recall    = tp / (tp + fn + 1e-9)
specificity = tn / (tn + fp + 1e-9)
f1 = 2 * precision * recall / (precision + recall + 1e-9)
print(f"Precision: {precision:.3f}  Recall: {recall:.3f}  Specificity: {specificity:.3f}  F1: {f1:.3f}")

# 詳細レポート（多クラスでもOK）
print("\nClassification report:\n", classification_report(y_true, y_pred, digits=3))



BEST test_acc=0.7353 (epoch 12)
Confusion matrix (counts):
 [[15  2]
 [ 7 10]]
Confusion matrix (row-normalized):
 [[0.882 0.118]
 [0.412 0.588]]
Precision: 0.833  Recall: 0.588  Specificity: 0.882  F1: 0.690

Classification report:
               precision    recall  f1-score   support

           0      0.682     0.882     0.769        17
           1      0.833     0.588     0.690        17

    accuracy                          0.735        34
   macro avg      0.758     0.735     0.729        34
weighted avg      0.758     0.735     0.729        34



In [87]:
# ep=100 のモデルで評価
state100 = torch.load("epoch100.pt", map_location=device)
model.load_state_dict(state100)

y_true100, y_pred100 = predict_all(model, test_loader, device)
from sklearn.metrics import confusion_matrix, classification_report
cm100 = confusion_matrix(y_true100, y_pred100, labels=[0,1])
cm_norm100 = confusion_matrix(y_true100, y_pred100, labels=[0,1], normalize="true")
print("\n[Epoch 100] Confusion (counts):\n", cm100)
print("Confusion matrix (row-normalized):\n", np.round(cm_norm100, 3))
print("[Epoch 100] Report:\n", classification_report(y_true100, y_pred100, digits=3))



[Epoch 100] Confusion (counts):
 [[13  4]
 [ 8  9]]
Confusion matrix (row-normalized):
 [[0.765 0.235]
 [0.471 0.529]]
[Epoch 100] Report:
               precision    recall  f1-score   support

           0      0.619     0.765     0.684        17
           1      0.692     0.529     0.600        17

    accuracy                          0.647        34
   macro avg      0.656     0.647     0.642        34
weighted avg      0.656     0.647     0.642        34

